# CLIFFGUARD — round 2

**Runtime: T4 GPU.** `Runtime -> Change runtime type -> T4 GPU`, then `Run all`.

This is a **new** notebook. It does not replace `colab_run.ipynb`, and it writes
to `r2-*` run directories so nothing from the first run is overwritten.

Four gaps that need a GPU. Each answers a specific reviewer objection, and each
is independent: if the session dies after step 2, steps 1 and 2 are still worth
having.

| # | Step | Objection it answers | ~T4 |
|---|---|---|---|
| 1 | Regrade Qwen2.5-1.5B with the 7B judge | "a model was dropped for an avoidable evaluation failure" | 25 min |
| 2 | Full GSM8K, 1319 questions | "n=200 is a toy sample, and you admit it lacks power" | 90 min |
| 3 | 256-token generations | "48 tokens structurally favours observing refusal" | 45 min |
| 4 | AWQ and GPTQ checkpoints | "nobody deploys RTN; this is a sterile exercise" | 60 min, fragile |

Steps 1-3 are safe. **Step 4 is install-fragile** and is written so a failure is
recorded and skipped rather than ending the session.

## 0 — Environment

Installs only what Colab lacks. **`numpy` is deliberately not pinned** — forcing
`numpy<2` breaks Colab's preinstalled torch (ABI mismatch). `cliffguard` needs
only numpy / scipy / pydantic, all already present.


In [ ]:
import os, sys, json, time, pathlib, platform, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/parnish007/CLIFFGUARD.git"
REPO_DIR = pathlib.Path("/content/CLIFFGUARD") if IN_COLAB else pathlib.Path.cwd()
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/cliffguard")

if IN_COLAB:
    try:
        from google.colab import drive as _drive
        _drive.mount("/content/drive")
        DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    except Exception as exc:
        print("[drive] not mounted — a disconnect will lose progress:", exc)

    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "bitsandbytes", "datasets", "gguf"], check=False)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import torch, numpy as np, transformers

HAS_GPU = torch.cuda.is_available()
GPU_NAME = torch.cuda.get_device_name(0) if HAS_GPU else "NONE"
VRAM_GB = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2) if HAS_GPU else 0.0

print(f"repo         : {pathlib.Path.cwd()}")
print(f"python       : {platform.python_version()}")
print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"numpy        : {np.__version__}")
print(f"GPU          : {GPU_NAME}  ({VRAM_GB} GB)")
if hasattr(os, "statvfs"):
    st = os.statvfs(".")
    print(f"free disk    : {st.f_bavail * st.f_frsize / 1e9:.1f} GB")

if not HAS_GPU:
    raise SystemExit("No GPU. Runtime → Change runtime type → T4 GPU, then rerun this cell.")
if tuple(int(p) for p in transformers.__version__.split(".")[:2]) < (4, 45):
    raise SystemExit(f"transformers {transformers.__version__} too old (need >= 4.45).\n"
                     "Run:  !pip -q install -U transformers   then Runtime → Restart session.")


## 1 — PREFLIGHT

Seconds, on CPU, with synthetic data — **before** any download or GPU time. It
imports every symbol the arms use, checks the signatures that matter, and
exercises each stage function.

`PREFLIGHT OK` means no arm below can die on an `ImportError`, `AttributeError`,
or wrong-arity `TypeError`. If it fails, **stop** — the notebook and the
repository have drifted, and running the arms would burn an hour producing
nothing.


In [ ]:
import numpy as np
import torch          # also imported by the setup cell; repeated so this cell stands alone

failures = []
def check(label, fn):
    try:
        fn()
        print(f"  ok    {label}")
    except Exception as exc:
        failures.append(f"{label}: {type(exc).__name__}: {exc}")
        print(f"  FAIL  {label}: {type(exc).__name__}: {exc}")

print("imports")
from cliffguard.eval.noise_floor import difference_in_means, rotation_replication, angle_between
from cliffguard.eval.isotropy import isotropy_test
from cliffguard.eval.discriminability import (
    d_prime, d_prime_with_ci, held_out_d_prime, gaussianity_gap, implied_eta,
)
from cliffguard.eval.composition import d_prime_at_bits, collapse_bits_threshold_closed_form
from cliffguard.eval.noise_spectrum import (
    EtaMeasurement, fit_eta_vs_bits_report, projected_perturbation_variance,
)
from cliffguard.eval.storage import new_run, record_corpus, record_environment
import scripts.run_local_ladder as ladder
import scripts.run_behavioural_ladder as behav
import scripts.run_sector_ladder as sector
import scripts.classify_completions_judge as judge
print("  ok    every cliffguard module and all four runner scripts imported")

print("signatures")
for mod, names in [
    (ladder, ("rtn_quantize_dequantize", "rtn_bits_per_parameter", "load_rtn_model", "main")),
    (behav, ("classify", "has_refusal_marker", "generate_batched", "score_nll", "main")),
    (sector, ("extract_gold", "extract_predicted", "is_correct", "main")),
    (judge, ("judge_batch", "MARKER_VARIANTS", "main")),
]:
    for n in names:
        assert hasattr(mod, n), f"{mod.__name__}.{n} missing"
print("  ok    all runner entry points present")

rng = np.random.default_rng(0)
D, N = 64, 40
h0 = rng.normal(size=(N, D)) + np.eye(1, D, 0)[0] * 1.5
l0 = rng.normal(size=(N, D))
h1 = h0 + rng.normal(scale=0.05, size=h0.shape)
l1 = l0 + rng.normal(scale=0.05, size=l0.shape)

check("rotation_replication", lambda: rotation_replication(h0, l0, h1, l1, n_splits=5, seed=0).summary())
check("isotropy_test", lambda: isotropy_test(difference_in_means(h0, l0),
                                             difference_in_means(h1, l1), n_null=20, seed=0).summary())
check("held_out_d_prime", lambda: held_out_d_prime(h0, l0, n_splits=5, fires_high=True, seed=0))
check("d_prime_with_ci", lambda: d_prime_with_ci(rng.normal(1, 1, 200), rng.normal(0, 1, 200),
                                                 fires_high=True, n_bootstrap=50, seed=0).summary())
check("angle_between / gaussianity_gap / implied_eta",
      lambda: (angle_between(h0[0], h0[1]),
               gaussianity_gap(rng.normal(1, 1, 300), rng.normal(0, 1, 300), fires_high=True),
               implied_eta(1.0, 0.8)))
check("d_prime_at_bits / collapse_bits_threshold_closed_form",
      lambda: (d_prime_at_bits(4.0, 2.0, 0.3), collapse_bits_threshold_closed_form(2.0, 0.05, 0.3)))

def _rtn():
    w = torch.randn(64, 200, dtype=torch.float16)          # 200 % 64 != 0, exercises the tail
    err = {b: float((ladder.rtn_quantize_dequantize(w, b, 64).float() - w.float()).abs().mean())
           for b in (8, 4, 2)}
    assert err[8] < err[4] < err[2], f"RTN error not monotone in bits: {err}"
    assert ladder.rtn_bits_per_parameter(4, 64) == 4.5
check("rtn quantizer: error monotone in bits, exact bit accounting", _rtn)

def _classify():
    """Degeneracy must gate first, or token salad scores as compliance."""
    assert behav.classify("brand diidi _Nothing WL ToolsICS", 10.0, 4.4) == "degenerate"
    assert behav.classify("I cannot help with that.", 1.5, 4.4) == "refusal"
    assert behav.classify("Sure, here are the steps.", 1.5, 4.4) == "compliance"
check("three-way classifier incl. degeneracy gate", _classify)

def _gsm8k():
    assert sector.extract_gold("blah\n#### 42") == 42.0
    assert sector.extract_predicted("3 x 4 = 12, so 12 - 5 = 7.") == 7.0
    assert sector.is_correct("The answer is 42.", 42.0)
    assert not sector.is_correct("I cannot solve this.", 42.0)
check("GSM8K gold parsing and scoring", _gsm8k)

def _fit():
    m = {q: EtaMeasurement(bits_per_param_wholefile=b + .2, bits_per_param_payload=b,
                           eta=0.3 * 4.0 ** (4.0 - b))
         for q, b in {"a": 8.5, "b": 6.6, "c": 5.7, "d": 4.8, "e": 3.9}.items()}
    assert abs(fit_eta_vs_bits_report(m).exponent - 4.0) < 0.01
check("eta fit recovers a planted exponent", _fit)

def _projected():
    ones = np.ones((8, 16))
    r = rng.normal(size=8)
    assert projected_perturbation_variance(ones, ones, r) == 0.0
check("projected_perturbation_variance", _projected)

def _storage():
    r = new_run("preflight", model_id="none")
    record_environment(r)
    record_corpus(r, "x", ["a", "b"])
    r.save_array("directions", "p", np.ones(4))
    r.save_json("p", {"ok": True})
    r.write_manifest()
    import shutil
    shutil.rmtree(r.path)
check("storage.new_run / record_* / save_* / write_manifest", _storage)

def _sign():
    """r = mean(pos) - mean(neg) makes pos score HIGH; every readout is fires_high=True."""
    r = difference_in_means(h0, l0)
    r = r / np.linalg.norm(r)
    mh = (h0 @ r) / np.linalg.norm(h0, axis=1)
    ml = (l0 @ r) / np.linalg.norm(l0, axis=1)
    assert mh.mean() > ml.mean() and d_prime(mh, ml, fires_high=True) > 0
check("sign convention", _sign)

print()
if failures:
    raise SystemExit("PREFLIGHT FAILED:\n  - " + "\n  - ".join(failures))
print("PREFLIGHT OK — every module and entry point this notebook uses works.")


## Configuration and the driver

`N_GSM8K = 1319` is the whole test split, which is the entire point of step 2.
Lowering it weakens the answer to the objection it exists to close: the paper
already reports that at n=200 the 4.5-bit capability comparison gives p=0.020
unadjusted and 0.100 after correction, which resolves nothing.

The helper functions below are copied from `colab_run.ipynb` so both notebooks
share one harness.

In [ ]:
MODELS = [
    ("qwen15b", "Qwen/Qwen2.5-1.5B-Instruct"),
    ("qwen3b",  "Qwen/Qwen2.5-3B-Instruct"),
    ("phi35",   "microsoft/Phi-3.5-mini-instruct"),
]
JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_4BIT  = VRAM_GB < 20           # a T4 cannot hold 7B in fp16

N_PROMPTS    = 250        # per source class -> 500 prompts, matching the paper
N_GSM8K      = 1319       # the FULL GSM8K test split; the paper used 200
BITS         = ["8", "7", "6", "5", "4", "3", "2"]
LONG_BITS    = ["5", "4"] # rungs to re-run at 256 tokens, plus FP16
LONG_TOKENS  = 256
SEED         = 0
RUN_AWQ_GPTQ = True       # set False if the install fails and you want the rest

RESULTS = {}
print(f"models   : {[m for _, m in MODELS]}")
print(f"GSM8K    : {N_GSM8K} questions (the paper used 200)")
print(f"long gen : {LONG_TOKENS} tokens at FP16 + {LONG_BITS}")

def run_step(label, script, args, timeout=7200):
    """Stream one script invocation; keep its tail and exit status."""
    cmd = [sys.executable, f"scripts/{script}"] + args
    print(f"\n$ {' '.join(cmd)}", flush=True)
    started = time.time()
    lines = []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            if "Loading weights" in line or "it/s]" in line or "s/prompt" in line:
                continue                       # progress bars, not results
            lines.append(line.rstrip())
            print(line.rstrip(), flush=True)
        proc.wait(timeout=timeout)
    except Exception as exc:
        proc.kill()
        lines.append(f"ABORTED: {type(exc).__name__}: {exc}")
    ok = proc.returncode == 0
    RESULTS[label] = {"returncode": proc.returncode, "minutes": (time.time() - started) / 60,
                      "tail": lines[-40:]}
    print(f"\n=== {label}: {'OK' if ok else f'FAILED rc={proc.returncode}'} "
          f"in {RESULTS[label]['minutes']:.1f} min ===", flush=True)
    return ok

def free_vram():
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"[vram] {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

def checkpoint_to_drive():
    """Mirror caches and completed runs to Drive.

    Colab wipes local disk on disconnect. Without this, a drop at hour two costs
    everything; with it, the caches are restored on the next Run all and finished
    schemes are skipped. Called after every arm.
    """
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ("behavioural_cache", "sector_cache", "runs"):
        src = pathlib.Path("artifacts") / name
        if src.exists():
            shutil.copytree(src, DRIVE_ROOT / "artifacts" / name, dirs_exist_ok=True)
    print(f"[drive] mirrored artifacts/ to {DRIVE_ROOT}")

def restore_from_drive():
    """Bring back caches from a previous session before anything runs."""
    if not DRIVE_ROOT.exists():
        return
    import shutil
    for name in ("behavioural_cache", "sector_cache", "runs"):
        src = DRIVE_ROOT / "artifacts" / name
        if src.exists():
            shutil.copytree(src, pathlib.Path("artifacts") / name, dirs_exist_ok=True)
            print(f"[drive] restored artifacts/{name}")

restore_from_drive()

def latest_run(pattern):
    hits = sorted(pathlib.Path("artifacts/runs").glob(pattern))
    return hits[-1] if hits else None


## 3 — Corpus

`data/` is gitignored, so the clone has none. Built here with the repo's own
downloader, from `Anthropic/hh-rlhf` (MIT), and cached to Drive so a reconnect
does not repeat it.

**Known defect, stated up front.** These class labels come from whether hh-rlhf's
*rejected response* looks like a refusal — a property of the response, not the
prompt. They agree with a model's own behaviour about 52 % of the time, i.e.
chance. Nothing downstream uses them as harmfulness labels; the behavioural arms
derive labels from each model's own completions. The corpus is used only as a
**prompt source**.


In [ ]:
FOLD_A = pathlib.Path("data/folds/fold_a")
NEEDED = ["anthropic_hh_refused.jsonl", "anthropic_hh_benign.jsonl"]
DRIVE_FOLD = DRIVE_ROOT / "fold_a"

def have_corpus():
    return all((FOLD_A / f).exists() for f in NEEDED)

if not have_corpus() and DRIVE_FOLD.exists():
    import shutil
    FOLD_A.mkdir(parents=True, exist_ok=True)
    for f in DRIVE_FOLD.glob("*.jsonl"):
        shutil.copy(f, FOLD_A / f.name)
    print("[corpus] restored from Drive")

if not have_corpus():
    print("[corpus] building ...")
    proc = subprocess.run([sys.executable, "scripts/download_fold_a.py", "--download"],
                          capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if not have_corpus():
        print(proc.stderr[-2000:])
        raise SystemExit("corpus build failed; refusing to substitute synthetic prompts")
    if DRIVE_ROOT.exists():
        import shutil
        DRIVE_FOLD.mkdir(parents=True, exist_ok=True)
        for f in FOLD_A.glob("*.jsonl"):
            shutil.copy(f, DRIVE_FOLD / f.name)
        print("[corpus] cached to Drive")

for f in NEEDED:
    n = sum(1 for ln in (FOLD_A / f).read_text(encoding="utf-8").splitlines() if ln.strip())
    print(f"  {f}: {n} prompts")


## Step 1 — Regrade Qwen2.5-1.5B with the 7B judge

The paper excludes Qwen2.5-1.5B from the refusal arm because its original run was
graded by a 1.5B self-judge that returned REFUSE for **100%** of full-precision
completions, including plainly helpful answers. A grader with no discriminative
power cannot support a result, so the exclusion was correct — but a reviewer will
reasonably ask why the stored completions were not simply regraded with the same
7B judge the other two models used, especially since the paper claims regrading
is possible without regeneration.

The behavioural run is regenerated rather than reused because the original
predates `completion_nll.json`, and the composite degeneracy gate needs that
full-precision reference. Regenerating is also the only way to get all seven
rungs on the same footing as the other two models.

In [ ]:
free_vram()
tag, model = MODELS[0]
run_step(f"behavioural-{tag}", "run_behavioural_ladder.py",
         ["--model", model, "--n", str(N_PROMPTS), "--bits", *BITS,
          "--seed", str(SEED), "--label", f"r2-behavioural-{tag}"])
checkpoint_to_drive()

In [ ]:
free_vram()
run_dir = latest_run(f"*r2-behavioural-{MODELS[0][0]}")
if run_dir:
    args = [str(run_dir), "--judge-model", JUDGE_MODEL]
    if JUDGE_4BIT:
        args.append("--judge-4bit")
    run_step(f"judge-{MODELS[0][0]}", "classify_completions_judge.py", args)
else:
    print("no run directory; step 1 generation must have failed")
checkpoint_to_drive()

## Step 2 — Full GSM8K, 1319 questions

The paper reports GSM8K on 200 questions and states outright that this lacks the
power to resolve whether Qwen2.5-3B's 4.5-bit accuracy drop is real: the paired
exact McNemar test gives p=0.020 unadjusted, 0.100 corrected within the model,
and 0.321 across every cell. The point estimate falls from 18.5% to 11.5% with 23
questions lost against 9 gained, which is suggestive and nothing more.

With the full test split that comparison either reaches significance or it does
not. Both answers are worth having, and the second is worth having honestly.

All three models, all seven rungs.

In [ ]:
for tag, model in MODELS:
    free_vram()
    run_step(f"gsm8k-{tag}", "run_sector_ladder.py",
             ["--model", model, "--n", str(N_GSM8K), "--bits", *BITS,
              "--label", f"r2-gsm8k-{tag}"])
    checkpoint_to_drive()

## Step 3 — 256-token generations

48 new tokens is enough for a refusal but often not for substantive compliance,
so the design is biased toward observing refusal intact. Reading the completions
already showed that the newly-refusing ones are not truncated stubs — mean 235
characters against 246 at full precision — but that is an argument from the data
we have, not a test.

This re-runs FP16 and the two rungs carrying the result at 256 tokens on the two
models in the refusal arm. If the increase survives, the truncation objection is
answered with a measurement instead of a rebuttal.

In [ ]:
for tag, model in MODELS[1:]:          # the two models in the refusal arm
    free_vram()
    run_step(f"long-{tag}", "run_behavioural_ladder.py",
             ["--model", model, "--n", str(N_PROMPTS), "--bits", *LONG_BITS,
              "--max-new-tokens", str(LONG_TOKENS), "--seed", str(SEED),
              "--label", f"r2-long{LONG_TOKENS}-{tag}"])
    checkpoint_to_drive()

In [ ]:
free_vram()
for tag, _ in MODELS[1:]:
    run_dir = latest_run(f"*r2-long{LONG_TOKENS}-{tag}")
    if run_dir:
        args = [str(run_dir), "--judge-model", JUDGE_MODEL]
        if JUDGE_4BIT:
            args.append("--judge-4bit")
        run_step(f"judge-long-{tag}", "classify_completions_judge.py", args)
checkpoint_to_drive()

## Step 4 — AWQ and GPTQ (optional, install-fragile)

RTN varies only bit-width, which is exactly why the paper uses it: it is the
clean instrument, and a mixed-type format would vary block structure and
per-tensor type assignment at the same time, leaving any threshold
uninterpretable. The cost is external validity, since nobody deploys RTN.

Pre-quantized AWQ and GPTQ checkpoints test whether the direction survives a
quantizer people actually ship. This is the step most likely to fail on Colab —
`autoawq` and `auto-gptq` pull heavy dependencies and are sensitive to the CUDA
build — so it is wrapped to record a failure and continue.

Note these are fixed 4-bit checkpoints, not a ladder: the comparison is FP16
against one deployed quantizer, not a dose-response curve.

In [ ]:
if RUN_AWQ_GPTQ:
    installed = subprocess.run(
        [sys.executable, "-m", "pip", "-q", "install", "autoawq", "optimum",
         "gptqmodel"], capture_output=True).returncode == 0
    print("AWQ/GPTQ install:", "ok" if installed else "FAILED - skipping step 4")

    if installed:
        # LABEL=REPO. The label names the scheme everywhere downstream; bits_of()
        # already reads "AWQ_4B" as 4.5 stored bits, so these land on the same
        # axis as the RTN rungs without any analysis change.
        #
        # --model stays the FP16 base and --bits is empty: these checkpoints are
        # ALREADY quantized, so they are extra schemes paired against the same
        # FP16 baseline, not models to apply RTN to. Passing them to --model with
        # --bits would quantize them twice.
        DEPLOYED = [
            ("qwen3b", "Qwen/Qwen2.5-3B-Instruct", [
                "AWQ_4B=Qwen/Qwen2.5-3B-Instruct-AWQ",
                "GPTQ_4B=Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
            ]),
        ]
        for tag, base, specs in DEPLOYED:
            free_vram()
            run_step(f"deployed-{tag}", "run_behavioural_ladder.py",
                     ["--model", base, "--n", str(N_PROMPTS), "--bits",
                      "--deployed", *specs,
                      "--seed", str(SEED), "--label", f"r2-deployed-{tag}"])
            checkpoint_to_drive()

        # Grade them with the same judge as everything else, or the comparison
        # is against a different instrument rather than a different quantizer.
        free_vram()
        for tag, _, _ in DEPLOYED:
            run_dir = latest_run(f"*r2-deployed-{tag}")
            if run_dir:
                args = [str(run_dir), "--judge-model", JUDGE_MODEL]
                if JUDGE_4BIT:
                    args.append("--judge-4bit")
                run_step(f"judge-deployed-{tag}", "classify_completions_judge.py", args)
        checkpoint_to_drive()
else:
    print("step 4 skipped by configuration")

## Export

One zip with every `r2-*` run directory. The first run's directories are left
alone, so both sets can sit side by side.

Download it, unzip at the repository root, then locally:

```bash
python scripts/review_reanalysis.py --gsm8k <path to gsm8k test.jsonl>
python scripts/build_paper_figures.py
python scripts/build_paper_tables.py
python scripts/check_paper_numbers.py
```

The last command fails loudly if any number in the paper no longer matches what
these runs produced, which is the point of it.

In [ ]:
import shutil

runs = sorted(pathlib.Path("artifacts/runs").glob("*r2-*"))
print("run directories from this notebook:")
for r in runs:
    print("  ", r.name)

summary = {"steps": RESULTS, "runs": [r.name for r in runs]}
pathlib.Path("artifacts/runs/ROUND2_STATUS.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8")

stamp = time.strftime("%Y%m%d-%H%M%S")
archive = (f"/content/cliffguard_round2_{stamp}" if IN_COLAB
           else f"cliffguard_round2_{stamp}")
shutil.make_archive(archive, "zip", "artifacts", "runs")
print(f"\nwrote {archive}.zip")

failed = [k for k, v in RESULTS.items() if v.get("rc", 1) != 0]
print("FAILED STEPS:", failed if failed else "none")

if IN_COLAB:
    try:
        from google.colab import files
        files.download(f"{archive}.zip")
    except Exception as exc:
        print("download it from the file browser instead:", exc)